> ## SOLUTION / ANSWER KEY &mdash; Lab 8.5
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-05-parallel-fanout.ipynb`](../lab-05-parallel-fanout.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 8.5 &mdash; Parallel Fan-Out

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Run every specialist on the SAME customer ticket at once
- Collect each result tagged with the agent that produced it
- Survive one agent failing -- fault tolerance in a fan-out

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Parallel — fan-out for coverage & speed](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = "/tmp/biaa-lab-08-05"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
In the **parallel** (fan-out) shape, several agents work the **same input** at once and their outputs are
combined (deck slide 10). For a support ticket you fan it out to the **billing, tech and policy** specialists
together &mdash; each an independent lens, so between them they catch what one alone would miss. Two practical
rules: keep each result **tagged with the agent** that produced it (you must know who said what), and make
the fan-out **fault-tolerant** &mdash; if one specialist is down, the others still return. Fan-out then
creates a new problem &mdash; **several outputs, and you need one** &mdash; which is the decision-making you
build next.

In [ ]:
# Three specialists, each a different lens on the SAME ticket -- and one of them is currently DOWN.
def billing_agent(t):
    return "duplicate charge on 4471" if "charg" in t.lower() else "no billing issue"
def tech_agent(t):
    return "matches BUG-231" if "crash" in t.lower() else "no tech issue"
def policy_agent(t):
    raise RuntimeError("policy service unavailable")   # this specialist is DOWN -> the fan-out must survive it
SPECIALISTS = {"billing": billing_agent, "tech": tech_agent, "policy": policy_agent}
print("fan-out targets:", list(SPECIALISTS))

## Build it
Complete `fan_out`: run every specialist on the **same** ticket, tag each result by agent, and keep going
when one raises.

In [ ]:
def fan_out(ticket, specialists):
    results = {}
    for name, agent in specialists.items():
        try:
            results[name] = agent(ticket)
        except Exception as e:
            results[name] = f"ERROR: {type(e).__name__}"
    return results

try:
    out = fan_out("charged twice and the app keeps crashing", SPECIALISTS)
    for name, res in out.items():
        print(f"{name:8}: {res}")
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- Every specialist saw the **same ticket**; each result is **tagged** with who produced it &mdash; you always know who said what.
- The **down** `policy` agent yields an `"ERROR: ..."` marker instead of crashing the fan-out &mdash; the survivors still returned findings.
- You now hold **several** results and need **one** &mdash; that convergence (vote / synthesise) is the rest of the module.

## Your turn (open task &mdash; no grader)
Add a fourth specialist that returns a *contradicting* billing verdict (e.g. `"no refund due"`), then re-run
the fan-out. **What good looks like:** all four run, each tagged, the down agent still degrades to a marker &mdash;
and you now have a genuine **conflict** to resolve with a vote (Lab 8.7) or synthesis (Lab 8.9).

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
def billing_review(t):
    return "no refund due" if "charg" in t.lower() else "no billing issue"   # CONTRADICTS billing_agent

SPECIALISTS["billing_review"] = billing_review
out = fan_out("charged twice and the app keeps crashing", SPECIALISTS)
for name, res in out.items():
    print(f"{name:14}: {res}")            # all run, each tagged; policy still degrades to an ERROR marker

---
### Key takeaway
Fan-out buys coverage and speed -- latency is the slowest agent, not the sum -- and staying tagged + fault-tolerant means one agent going down doesn't take the team with it. But now you have several outputs and need one: that convergence is decision making, coming up.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>